# 02 - Créer notre tokenizer darija

Nous allons construire un tokenizer simple par mots, comprendre les mots inconnus, puis apprendre un tokenizer **ByteLevel BPE** sur les données préparées au notebook 01.

Le vocabulaire et les fusions seront appris sur notre partition d'entraînement. Aucun tokenizer préentraîné n'est chargé. La bibliothèque `tokenizers` fournit l'algorithme BPE ; le modèle GPT sera construit plus tard.

**Ce notebook est livré non exécuté.** La cellule intitulée « Apprendre les fusions » ne sera exécutée que lorsque tu la lanceras toi-même. Les phrases courtes écrites dans ce notebook sont des exemples pédagogiques, pas des données AtlasIA inventées.

In [ ]:
import hashlib
import json
import re
import unicodedata
from pathlib import Path

import tokenizers
from tokenizers import Tokenizer, decoders, models, normalizers
from tokenizers import pre_tokenizers, trainers

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / 'README.md').is_file() and (p / 'notebooks').is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError('Ouvre ce notebook depuis le projet darija-llm-lab.')
PROCESSED_DIR = ROOT / 'data' / 'processed'
TOKENIZER_DIR = ROOT / 'artifacts' / 'tokenizer'
VOCAB_SIZE = 2000
MIN_FREQUENCY = 2
SPECIAL_TOKENS = ['[UNK]', '[PAD]', '[BOS]', '[EOS]']
print('Version Tokenizers :', tokenizers.__version__)
print('Vocabulaire cible :', VOCAB_SIZE)

## 1. Tokens, identifiants et vocabulaire

Un **token** est un morceau de texte ; un **identifiant** est son numéro dans le vocabulaire. Ce numéro n'exprime ni le sens ni l'importance du token. Le futur GPT transformera ces numéros en vecteurs appelés embeddings.

Pour le premier essai, nous découpons les suites de caractères de mot et isolons la ponctuation. `encode` convertit les unités en identifiants ; `decode` effectue l'opération inverse.

In [ ]:
def normalize(text):
    return unicodedata.normalize('NFC', text)

class WordTokenizer:
    def __init__(self, texts):
        pieces = {
            piece for text in texts
            for piece in re.findall(r'\w+|[^\w\s]', normalize(text))
        }
        self.vocab = {'[UNK]': 0}
        self.vocab.update({piece: i + 1 for i, piece in enumerate(sorted(pieces))})
        self.inverse = {i: piece for piece, i in self.vocab.items()}

    def encode(self, text):
        pieces = re.findall(r'\w+|[^\w\s]', normalize(text))
        return [self.vocab.get(piece, 0) for piece in pieces]

    def decode(self, ids):
        return ' '.join(self.inverse[i] for i in ids)

toy = WordTokenizer(['salam, labas?'])
print('Vocabulaire :', toy.vocab)
print('Identifiants :', toy.encode('salam, labas?'))
print('Décodage :', toy.decode(toy.encode('salam, labas?')))
print('Mot inconnu :', toy.encode('salam khouya'))

La version par mots remplace une unité inconnue par `[UNK]` et réinsère des espaces au décodage. Elle peut donc perdre le mot original et la mise en forme. Elle sert à comprendre le mécanisme, pas à préserver tout texte Unicode.

**Exercice :** prévois le résultat de `toy.encode('salam khouya')`. Pourquoi le décodage ne peut-il pas retrouver `khouya` ?

## 2. L'idée de BPE

Exemple simplifié sur caractères : deux documents `abab` et un document `ab`. La paire `(a,b)` apparaît cinq fois ; `(b,a)` apparaît deux fois. La première fusion crée `ab`. Les documents deviennent `[ab, ab]`, `[ab, ab]` et `[ab]`. On recompte ensuite les paires.

L'apprentissage mémorise des unités et un ordre de fusions. L'encodage d'un nouveau texte utilise ensuite ces règles sans les réapprendre. BPE apprend des fréquences, pas des relations de sens.

Notre variante **ByteLevel** part des octets UTF-8. Une lettre arabe peut occuper plusieurs octets : octet, caractère et token ne sont pas la même chose. L'alphabet initial de 256 octets permet de représenter de nouveaux textes sans perdre les symboles ordinaires.

## 3. Recharger les partitions du notebook 01

Nous vérifions les hashes des textes et l'absence de doublons exacts entre les partitions. Le tokenizer sera appris **uniquement sur `train_texts`**. La validation servira aux mesures après apprentissage.

In [ ]:
def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f'{path} absent : exécute le notebook 01.')
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()
            if line.strip()]
    for row in rows:
        actual_hash = hashlib.sha256(row['text'].encode('utf-8')).hexdigest()
        if actual_hash != row['sha256']:
            raise ValueError('Le texte a changé depuis sa préparation.')
    return rows

train_path = PROCESSED_DIR / 'train.jsonl'
valid_path = PROCESSED_DIR / 'validation.jsonl'
train_rows = read_jsonl(train_path)
valid_rows = read_jsonl(valid_path)
train_texts = [row['text'] for row in train_rows]
valid_texts = [row['text'] for row in valid_rows]
if not train_texts or not valid_texts:
    raise ValueError('Les deux partitions doivent contenir des documents.')
assert set(train_texts).isdisjoint(valid_texts)
corpus_manifest = json.loads(
    (PROCESSED_DIR / 'corpus_manifest.json').read_text(encoding='utf-8')
)
print('Entraînement :', len(train_texts), '| Validation :', len(valid_texts))

## 4. Définir le tokenizer avant son apprentissage

- **NFC** : même convention de normalisation qu'au notebook 01.
- **ByteLevel** : représentation des octets et pré-découpage ; aucun espace artificiel ajouté au début.
- **BpeTrainer** : apprend les fusions à partir des textes. La taille cible inclut l'alphabet et les tokens spéciaux.
- **Décodeur ByteLevel** : reconstruit le texte normalisé à partir d'une séquence d'identifiants.

Tokens spéciaux : `[UNK]` pour une unité inconnue, `[PAD]` pour compléter une séquence, `[BOS]` pour un début et `[EOS]` pour une fin. Les réserver ne les insère pas automatiquement. Dans notre pipeline, EOS sera ajouté explicitement au notebook 03.

In [ ]:
def build_bpe(vocab_size=2000, min_frequency=2):
    if vocab_size < 256 + len(SPECIAL_TOKENS):
        raise ValueError('Il faut couvrir les 256 octets et les tokens spéciaux.')
    tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=min_frequency,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        special_tokens=SPECIAL_TOKENS, show_progress=True,
    )
    return tokenizer, trainer

## 5. Apprendre les fusions

**Cette cellule lance l'apprentissage quand tu l'exécutes.** Vérifie d'abord le corpus et les paramètres. Elle n'apprend ni sur les phrases inventées ci-dessus ni sur la validation.

Dans Tokenizers 0.22.1, `encode_special_tokens=True` permet de traiter les chaînes littérales comme `[EOS]` présentes dans un texte comme du texte ordinaire. Cela évite de les confondre avec les marqueurs que nous ajouterons par leur identifiant.

In [ ]:
tokenizer, trainer = build_bpe(VOCAB_SIZE, MIN_FREQUENCY)
tokenizer.train_from_iterator(
    train_texts, trainer=trainer, length=len(train_texts)
)
tokenizer.encode_special_tokens = True
print('Vocabulaire réel :', tokenizer.get_vocab_size())
print('Tokens spéciaux :', {s: tokenizer.token_to_id(s) for s in SPECIAL_TOKENS})

## 6. Voir les tokens et vérifier le décodage

`encoding.tokens` affiche les représentations internes de ByteLevel : certaines peuvent paraître étranges. Pour retrouver le texte, décode la **séquence complète**, pas chaque token isolé. Un token peut contenir seulement une partie des octets d'une lettre arabe.

L'invariant attendu est `decode(encode(texte)) == NFC(texte)`. Il porte sur le texte normalisé, car la normalisation peut changer sa représentation Unicode.

In [ ]:
examples = [
    'سلام، لاباس؟', 'salam, labas? wach kolchi mzyan?',
    'ghadi nji demain.', 'cafe\u0301', 'salam  labas\n🙂',
    'texte littéral [UNK] [PAD] [BOS] [EOS]', '',
]
unk_id = tokenizer.token_to_id('[UNK]')
for text in examples + valid_texts:
    encoding = tokenizer.encode(text, add_special_tokens=False)
    decoded = tokenizer.decode(encoding.ids, skip_special_tokens=False)
    assert decoded == normalize(text), (text, decoded)
    assert unk_id not in encoding.ids
print('Allers-retours validés :', len(examples) + len(valid_texts))

text = examples[0]
encoding = tokenizer.encode(text, add_special_tokens=False)
print('Texte :', text)
print('Tokens internes :', encoding.tokens)
print('Identifiants :', encoding.ids)
print('Décodage :', tokenizer.decode(encoding.ids))

## 7. Mesurer la fragmentation sur la validation

Zéro `[UNK]` vérifie la couverture, pas la compréhension de la langue. Nous mesurons aussi le nombre de tokens pour 100 caractères. Ici, Python compte des points de code Unicode, pas nécessairement des caractères visibles.

Un vocabulaire plus grand peut produire des séquences plus courtes, mais agrandit la future table d'embeddings. Compare les choix sur les mêmes documents de validation ; ne prétends pas qu'une taille est optimale sans mesure.

In [ ]:
valid_encodings = [tokenizer.encode(text, add_special_tokens=False)
                   for text in valid_texts]
n_tokens = sum(len(item.ids) for item in valid_encodings)
n_chars = sum(len(normalize(text)) for text in valid_texts)
evaluation = {
    'validation_documents': len(valid_texts),
    'validation_tokens': n_tokens,
    'tokens_per_100_characters': round(100 * n_tokens / max(1, n_chars), 2),
    'unknown_tokens': sum(item.ids.count(unk_id) for item in valid_encodings),
    'roundtrip_documents_checked': len(valid_texts),
}
print(json.dumps(evaluation, indent=2, ensure_ascii=False))

## 8. Sauvegarder et recharger

Le fichier JSON conserve les composants du tokenizer, le vocabulaire et les fusions. Dans notre version de la bibliothèque, `encode_special_tokens=True` doit être **rétabli après rechargement**.

Les résultats vont dans `artifacts/tokenizer/`, exclu de Git. La présence de ce fichier après exécution matérialise le tokenizer réellement appris ; les notebooks livrés vides ne constituent pas un tokenizer déjà entraîné.

In [ ]:
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
tokenizer_path = TOKENIZER_DIR / 'tokenizer_darija.json'
tokenizer.save(str(tokenizer_path))
loaded = Tokenizer.from_file(str(tokenizer_path))
loaded.encode_special_tokens = True
for text in examples + valid_texts:
    assert loaded.encode(text).ids == tokenizer.encode(text).ids

training_manifest = {
    'dataset': corpus_manifest['dataset'],
    'corpus': corpus_manifest,
    'fit_partition': 'train', 'validation_used_for_training': False,
    'train_documents': len(train_texts), 'validation_documents': len(valid_texts),
    'train_jsonl_sha256': hashlib.sha256(train_path.read_bytes()).hexdigest(),
    'validation_jsonl_sha256': hashlib.sha256(valid_path.read_bytes()).hexdigest(),
    'requested_vocabulary': VOCAB_SIZE,
    'actual_vocabulary': tokenizer.get_vocab_size(),
    'min_frequency': MIN_FREQUENCY, 'normalization': 'NFC',
    'special_token_ids': {s: tokenizer.token_to_id(s) for s in SPECIAL_TOKENS},
    'literal_special_tokens_as_text': True,
    'tokenizers_version': tokenizers.__version__,
    'tokenizer_sha256': hashlib.sha256(tokenizer_path.read_bytes()).hexdigest(),
}
for name, content in [('training_manifest.json', training_manifest),
                      ('evaluation.json', evaluation)]:
    (TOKENIZER_DIR / name).write_text(
        json.dumps(content, ensure_ascii=False, indent=2), encoding='utf-8'
    )
print('Tokenizer et mesures sauvegardés dans :', TOKENIZER_DIR)

## Exercices

1. Pourquoi les identifiants du vocabulaire ne représentent-ils pas le sens des mots ?
2. Pourquoi faut-il au moins 260 entrées pour notre configuration initiale ?
3. Quelle différence entre `train_from_iterator` et `encode` ?
4. Essaie plus tard des tailles cibles de 500, 1 000 et 2 000 sur les mêmes partitions. Note la taille réelle et la fragmentation de validation.
5. Dans la page de BPE simplifié, quelle fusion devient possible après la création de `ab` ?

<details><summary>Corrigés des questions conceptuelles</summary>

1. Les identifiants sont des indices de dictionnaire. Le modèle apprendra ensuite ses propres vecteurs et relations de contexte.
2. 256 représentations d'octets et 4 tokens spéciaux. Les fusions agrandissent ensuite ce vocabulaire.
3. La première méthode apprend les fusions sur un corpus ; la seconde applique les règles existantes à un texte.
4. Les valeurs dépendent du corpus : ne pas inventer de résultats avant d'avoir exécuté les expériences.
5. La paire `(ab, ab)` apparaît deux fois et peut former `abab`.

</details>

**Suite :** [03 - Fenêtres glissantes](03_fenetres_glissantes.ipynb).

Références : [documentation Tokenizers](https://huggingface.co/docs/tokenizers/v0.20.3/en/quicktour), [ByteLevel](https://huggingface.co/docs/tokenizers/v0.20.3/en/api/pre-tokenizers), [apprentissage depuis un itérateur](https://huggingface.co/docs/tokenizers/v0.20.3/en/training_from_memory).